# Imports

In [1]:
import threading, time, json, re
import pandas as pd
from collections import deque
from queue import Queue, Empty
from openai import OpenAI
from tqdm.auto import tqdm

# Configuration

In [2]:
API_KEY    = "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"
MODEL      = "qwen/qwen3-next-80b-a3b-thinking"
BASE_URL   = "https://integrate.api.nvidia.com/v1"
INPUT_CSV  = "/home/mabdrabou/Desktop/NLP Project/mental_health.csv"
OUTPUT_CSV = "/home/mabdrabou/Desktop/NLP Project/qwen_labels_output.csv"

In [3]:
RPM_LIMIT  = 28
WINDOW_SEC = 60
REST_SEC   = 35

# Few Shot Prompt + LLM For Labeling

In [4]:
FEW_SHOT_EXAMPLES = """
--- Example 1
Message: "tell crush like her ive procrastinating months point im still unsure it get help pls"
Label: 0 (mentally well)
Reasoning: Person is nervous about confessing to a crush — a normal social situation. No depression, anxiety disorder, or distress signals.
Score: 2 | Rating: VERY_GOOD

--- Example 2
Message: "guys finally got girlfriend dumped toxic relationship negative effect wellbeing got help writing text dad said proud dealt could barely see nice theres loads posts people getting good relationships"
Label: 0 (mentally well)
Reasoning: Positive and forward-looking tone. Healthy coping and growth narrative after recovering from a toxic relationship.
Score: 1 | Rating: BEST

--- Example 3
Message: "worldppl cares give themi planet yrs one thing learned ppl care u something give them im tired wish born way care productive wanna connect ppl want cant connect anyone awful feel trapped"
Label: 1 (at risk)
Reasoning: Deep social isolation, feeling trapped, exhaustion, inability to connect. Multiple depression markers including hopelessness and emotional withdrawal.
Score: 6 | Rating: MODERATE_BAD

--- Example 4
Message: "dont know hold longer thought boyfriend could enough help hold possibly start meds even though love anything theres nothing going anywhere life fucking meaningless cant continue anymore"
Label: 1 (at risk)
Reasoning: Clear hopelessness, meaninglessness, suicidal ideation. Person feels unsupported and is struggling severely.
Score: 8 | Rating: VERY_BAD

--- Example 5
Message: "nothing look forward life dont many reasons keep going feel like nothing keeps going next day makes want hang"
Label: 1 (at risk)
Reasoning: Directly expresses suicidal ideation. Complete absence of hope. Crisis-level message.
Score: 9 | Rating: VERY_VERY_BAD

--- Example 6
Message: "cant anymore tried kill twice wish id succeeded last summer months ago hospitalized could anything cant say really helped told pdoc therapist well two weeks ago sure telling people anymore"
Label: 1 (at risk)
Reasoning: Two prior attempts, regret at not succeeding, recent hospitalization, lost trust in professional help. Extreme and immediate crisis.
Score: 10 | Rating: WORST
"""

PROMPT_TEMPLATE = """\
You are a clinical NLP expert. Assess the mental state of the person and respond ONLY with a valid JSON object — no markdown, no explanation, nothing else.

Scoring: 1=BEST, 2=VERY_GOOD, 3=GOOD, 4=MODERATE_GOOD, 5=NEUTRAL, 6=MODERATE_BAD, 7=BAD, 8=VERY_BAD, 9=VERY_VERY_BAD, 10=WORST
Label 0 → not struggling → score 1-4. Label 1 → struggling → score 6-10.

Examples:
{few_shot}

Message: "{text}"
Label: {label}

Respond with ONLY this JSON and nothing else:
{{"reasoning": "2-3 sentences", "dominant_signals": ["signal1", "signal2"], "score": <1-10>, "rating": "<RATING>"}}"""


def build_prompt(text: str, label: int) -> str:
    return PROMPT_TEMPLATE.format(
        few_shot=FEW_SHOT_EXAMPLES,
        text=str(text),
        label=f"{label} ({'mentally well' if label == 0 else 'at risk'})",
    )


def extract_json(raw: str):
    """Strip think-blocks / fences, then parse the first valid JSON object."""
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL)
    raw = re.sub(r"```(?:json)?", "", raw).strip()

    required = {"reasoning", "dominant_signals", "score", "rating"}

    for match in re.finditer(r"\{[^{}]*\}", raw, re.DOTALL):
        try:
            data = json.loads(match.group())
            if required.issubset(data):
                data["score"] = int(data["score"])
                return data
        except (json.JSONDecodeError, ValueError):
            continue

    try:
        s, e = raw.rfind("{"), raw.rfind("}") + 1
        if s != -1 and e > s:
            data = json.loads(raw[s:e])
            data["score"] = int(data["score"])
            return data
    except Exception:
        pass

    return None


_request_times: list[float] = []

def rate_limited_wait():
    while True:
        now = time.time()
        while _request_times and now - _request_times[0] >= WINDOW_SEC:
            _request_times.pop(0)

        if len(_request_times) < RPM_LIMIT:
            _request_times.append(now)
            return

        sleep_for = max(REST_SEC, WINDOW_SEC - (now - _request_times[0]) + 1)
        print(f"\n  [rate-limit] sleeping {sleep_for:.0f}s …")
        time.sleep(sleep_for)


client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def label_row(text: str, label: int, max_retries: int = 5):
    """Call the API (sequential, single key) and return parsed JSON or None."""
    backoff = REST_SEC
    prompt  = build_prompt(text, label)

    for attempt in range(1, max_retries + 1):
        try:
            rate_limited_wait()

            stream = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.6,
                top_p=0.7,
                max_tokens=10_000,
                stream=True,
            )

            raw = ""
            for chunk in stream:
                delta = chunk.choices[0].delta if chunk.choices else None
                if delta and delta.content:
                    raw += delta.content

            result = extract_json(raw)
            if result:
                return result

            print(f"\n  [row] attempt {attempt}: JSON parse failed, retrying…")

        except Exception as exc:
            msg = str(exc)
            print(f"\n  [row] attempt {attempt} error: {msg[:120]}")
            if "429" in msg:
                time.sleep(backoff)
                backoff = min(backoff * 2, 120)

    return None


def label_dataframe(df: pd.DataFrame,
                    text_col: str = "text",
                    label_col: str = "label") -> pd.DataFrame:

    total = len(df)
    print(f"Labeling {total:,} rows sequentially with one API key …\n")

    reasoning_col   = []
    score_col       = []
    rating_col      = []
    signals_col     = []

    failed_indices  = []

    with tqdm(total=total, unit="row", colour="cyan") as pbar:
        for idx, row in df.iterrows():
            result = label_row(str(row[text_col]), int(row[label_col]))

            if result:
                reasoning_col.append(result["reasoning"])
                score_col.append(result["score"])
                rating_col.append(result["rating"])
                signals_col.append(", ".join(result["dominant_signals"]))
            else:
                reasoning_col.append(None)
                score_col.append(None)
                rating_col.append(None)
                signals_col.append(None)
                failed_indices.append(idx)

            pbar.update(1)

    print(f"\nPass 1 done — labeled: {total - len(failed_indices):,}  |  failed: {len(failed_indices):,}")

    if failed_indices:
        print(f"\nRetrying {len(failed_indices)} failed rows …")
        still_failed = []

        with tqdm(total=len(failed_indices), unit="row", colour="yellow") as pbar:
            for idx in failed_indices:
                pos = df.index.get_loc(idx)        # positional index in df
                result = label_row(str(df.loc[idx, text_col]),
                                   int(df.loc[idx, label_col]))
                if result:
                    reasoning_col[pos]  = result["reasoning"]
                    score_col[pos]      = result["score"]
                    rating_col[pos]     = result["rating"]
                    signals_col[pos]    = ", ".join(result["dominant_signals"])
                else:
                    still_failed.append(idx)
                pbar.update(1)

        print(f"Retry done — recovered: {len(failed_indices) - len(still_failed):,}  "
              f"|  still missing: {len(still_failed):,}")

    out = df.copy()
    out["qwen_reasoning"]        = reasoning_col
    out["qwen_score"]            = score_col
    out["qwen_rating"]           = rating_col
    out["qwen_dominant_signals"] = signals_col

    out.to_csv(OUTPUT_CSV, index=False)
    labeled = out["qwen_score"].notna().sum()
    print(f"\nSaved → {OUTPUT_CSV}  ({labeled:,}/{total:,} labeled)")
    return out

In [ ]:
if __name__ == "__main__":
    df = pd.read_csv(INPUT_CSV)
    df_labeled = label_dataframe(df, text_col="text", label_col="label")
    print(df_labeled.head())